In [1]:
#def rag(question):
    #search_results = search(question)
    #user_prompt = build_prompt(question, search_results)
    #return llm(user_prompt)

In [ ]:
# Load the data
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
# Convert the load data into an array of string
texts = [doc["question"]+" "+doc["answer"] for doc in documents]

In [12]:
# Chose the model for embedding
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
from tqdm.auto import tqdm

In [13]:
# Convert the data into vectores
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
  batchs = texts[i:i+batch_size]
  batch_vectors = model.encode(batchs)
  vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [14]:
import numpy as np
X = np.array(vectors)

In [29]:
# Creating the index
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [15]:
# Download rag_helper.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-29 17:40:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-29 17:40:33 (33.9 MB/s) - ‘rag_helper.py’ saved [2134/2134]



### From here we perform keyword search with `index`

In [17]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [18]:
import os
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [21]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [26]:
# Then use the RAGBase class:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model="llama-3.3-70b-versatile"
)

In [27]:
# Ask it a question:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

"Yes, you can still sign up for the program. If you're interested in receiving a certificate, you'll need to submit your project while submissions are still being accepted. Additionally, keep in mind that while homework is not mandatory, it's recommended for reinforcing concepts and can impact your rank on the leaderboard. You can join the course at any time, but be sure to review the requirements for certification."

### And here we perform vector search with the `vindex`
- We can't pass vindex to RAG as-is. Text search takes the query string directly, but vector search needs the query as a vector first. So we subclass RAGBase and override search to encode the query before searching.

In [28]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [30]:
# Using it
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
    model="llama-3.3-70b-versatile"
)

In [31]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up for the program even though it has already begun. According to the provided context, you can start learning and submitting homework while the form is open, without needing a confirmation email or registration check. However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.'